In [1]:
import warnings
warnings.filterwarnings("ignore")
                        
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

## Trips

In [2]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [3]:
daily_trips_by_operator = D1.aggregate_daily_trips_by_operator(sofi_trips)

In [4]:
# just add the before and after, set to zero, otherwise LA Metro Events 
# single point drops away from chart visibility
append_la_metro_events = pd.DataFrame({
    'service_date': ["2026-07-09", "2026-07-11"],
})
append_la_metro_events = append_la_metro_events.assign(
    service_date = pd.to_datetime(append_la_metro_events.service_date).dt.normalize(),
    schedule_name = "LA Metro Events Schedule",
    n_trips = 0
)

daily_trips_by_operator2 = pd.concat(
    [daily_trips_by_operator, append_la_metro_events], 
    axis=0, ignore_index=True
)

In [5]:
chart_utils.trip_chart_with_event_dates(
    daily_trips_by_operator2, wc_vars.sofi_dates
).properties(
    title= "Daily Trips by Operator during World Cup",
    width=500, height=300
)

alt.LayerChart(...)

## Stop Arrivals

In [6]:
sofi_stop_arrivals = D2.filter_fct_daily_scheduled_stops_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times
)

# Do this separately, because we need stop's pt geom
arrivals_by_event_type = D2.aggregate_by_event_type(sofi_stop_arrivals)

In [7]:
weekday_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekday"]
).rename(columns = {
    **{c: f"weekday_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

weekend_wide = D2.make_wide(
    arrivals_by_event_type[arrivals_by_event_type.day_type == "weekend"]
).rename(columns = {
    **{c: f"weekend_{c}" for c in ["daily_arrivals_event", "daily_arrivals_non_event", "change_daily_arrivals"]}
})

arrivals_wide = pd.merge(
    weekday_wide,
    weekend_wide,
    on = ["schedule_name", "stop_id", "stop_name"],
    how = "inner"
).pipe(D2.merge_in_stop_geom, sofi_stop_arrivals)

In [8]:
operator_df = (
    arrivals_wide
    .groupby(["schedule_name", "route_id_array", "stop_name"])
    .agg({
        "weekday_change_daily_arrivals": "sum",
        "weekend_change_daily_arrivals": "sum",
        "stop_id": "nunique"
    })
    .reset_index()
    .rename(columns = {"stop_id": "n_stop_ids"})
)

In [9]:
for i in sorted(operator_df.schedule_name.unique()):
    chart = chart_utils.weekday_weekend_chart_by_operator(operator_df, i)
    display(chart)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)

alt.HConcatChart(...)